In [1]:
# 🧩 Block 1 Code — verify environment, install libs to the CURRENT kernel, and set paths
# This cell is safe to run multiple times. It installs only missing packages.

import sys, subprocess, os, platform, importlib.util
from pathlib import Path

# -------- Project paths (edit if you ever change layout) --------
BASE_DIR = Path(r"C:\Users\murth\Desktop\nlpSession02\codeBase")
PROJ_DIR = BASE_DIR / "translation"
PROJ_DIR.mkdir(parents=True, exist_ok=True)
print(f"[OK] Project dir: {PROJ_DIR}")

# -------- Helper: install a package if missing --------
def ensure(pkg, extra_args=None):
    if importlib.util.find_spec(pkg.split("==")[0].split("[")[0]) is None:
        print(f"[INSTALL] {pkg}")
        cmd = [sys.executable, "-m", "pip", "install", pkg]
        if extra_args: cmd += extra_args
        subprocess.check_call(cmd)
    else:
        print(f"[SKIP] {pkg} already available")

# -------- Minimal, CPU-friendly stack --------
# Torch CPU wheel (works on Windows). If this fails, your pip will fallback to standard index CPU wheel.
# Note: we don't pin strict versions to keep compatibility, but you can if you prefer.
ensure("torch", extra_args=["--index-url", "https://download.pytorch.org/whl/cpu"])
ensure("transformers")
ensure("sentencepiece")
ensure("datasets")
ensure("sacrebleu")
ensure("bert-score")
ensure("evaluate")
ensure("accelerate")

# -------- Print versions and sanity checks --------
import torch, transformers, datasets, sacrebleu, evaluate
print("\n[VERSIONS]")
print("python        :", sys.version.split()[0])
print("platform      :", platform.platform())
print("torch         :", torch.__version__, "| cuda available?", torch.cuda.is_available())
print("transformers  :", transformers.__version__)
print("datasets      :", datasets.__version__)
print("sacrebleu     :", sacrebleu.__version__)
print("evaluate      :", evaluate.__version__)

# -------- Seed & small config for later blocks --------
CONFIG = {
    "seed": 42,
    "base_dir": str(BASE_DIR),
    "proj_dir": str(PROJ_DIR),
    "dataset": "iwslt2017",        # we'll subset de->en later
    "lang_pair": "de-en",
    "train_size": 5000,
    "valid_size": 1000,
    "test_size": 1000,
    "bpe_vocab": 8000,             # sentencepiece vocab size
    "max_len": 40,                  # cap to keep CPU fast
    "lstm_hidden": 256,
    "lstm_layers": 2,
    "epochs": 3                     # 3 for class; 5 if you have time
}
print("\n[CONFIG]", CONFIG)

# -------- Save a tiny config file for continuity --------
cfg_path = PROJ_DIR / "mt_config.json"
with open(cfg_path, "w", encoding="utf-8") as f:
    import json; json.dump(CONFIG, f, indent=2)
print(f"[OK] Wrote config to: {cfg_path}")


[OK] Project dir: C:\Users\murth\Desktop\nlpSession02\codeBase\translation
[SKIP] torch already available
[SKIP] transformers already available
[INSTALL] sentencepiece
[SKIP] datasets already available
[INSTALL] sacrebleu
[INSTALL] bert-score
[INSTALL] evaluate
[SKIP] accelerate already available

[VERSIONS]
python        : 3.10.11
platform      : Windows-10-10.0.26100-SP0
torch         : 2.8.0+cpu | cuda available? False
transformers  : 4.56.1
datasets      : 2.19.0
sacrebleu     : 2.5.1
evaluate      : 0.4.5

[CONFIG] {'seed': 42, 'base_dir': 'C:\\Users\\murth\\Desktop\\nlpSession02\\codeBase', 'proj_dir': 'C:\\Users\\murth\\Desktop\\nlpSession02\\codeBase\\translation', 'dataset': 'iwslt2017', 'lang_pair': 'de-en', 'train_size': 5000, 'valid_size': 1000, 'test_size': 1000, 'bpe_vocab': 8000, 'max_len': 40, 'lstm_hidden': 256, 'lstm_layers': 2, 'epochs': 3}
[OK] Wrote config to: C:\Users\murth\Desktop\nlpSession02\codeBase\translation\mt_config.json


### 📦 Block 2 — Data Load & Split (IWSLT2017 de→en) — **FIXED CONFIG**

**Change:** use `load_dataset("iwslt2017", "iwslt2017-de-en")`.

**What this does:**  
- Read `mt_config.json`.  
- Load IWSLT2017 **de→en** with the correct builder config.  
- Normalize to `src`/`tgt`, length-filter, deterministic shuffle, subsample to 5k/1k/1k.  
- Save JSONL files into `translation/artifacts/` and print previews + stats.


In [3]:
# --- Block 2 Code (Fixed): Data Load & Split (IWSLT2017 de→en) ---
from pathlib import Path
import json, random
from typing import List, Dict
import datasets

# ---- Paths & config ----
BASE_DIR = Path(r"C:\Users\murth\Desktop\nlpSession02\codeBase")
PROJ_DIR = BASE_DIR / "translation"
ARTIFACTS = PROJ_DIR / "artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)

cfg_path = PROJ_DIR / "mt_config.json"
assert cfg_path.exists(), f"Config not found: {cfg_path}. Run Block 1 first."
with open(cfg_path, "r", encoding="utf-8") as f:
    CONFIG = json.load(f)

SEED     = int(CONFIG.get("seed", 42))
MAX_LEN  = int(CONFIG.get("max_len", 40))
TRN_N    = int(CONFIG.get("train_size", 5000))
VAL_N    = int(CONFIG.get("valid_size", 1000))
TST_N    = int(CONFIG.get("test_size", 1000))
LANGPAIR = CONFIG.get("lang_pair", "de-en")
SRC_LANG, TGT_LANG = LANGPAIR.split("-")
random.seed(SEED)

print("[CONFIG]", CONFIG)

# ---- Load dataset (correct builder config) ----
print("[LOAD] datasets.load_dataset('iwslt2017', 'iwslt2017-de-en')")
raw = datasets.load_dataset("iwslt2017", "iwslt2017-de-en")

# keys: 'train', 'validation', 'test'; texts under ex['translation']['de'/'en']
def normalize_split(ds: datasets.Dataset) -> List[Dict[str, str]]:
    rows = []
    for ex in ds:
        tr = ex.get("translation", {})
        src = (tr.get(SRC_LANG) or "").strip()
        tgt = (tr.get(TGT_LANG) or "").strip()
        if src and tgt:
            rows.append({"src": src, "tgt": tgt})
    return rows

train_rows = normalize_split(raw["train"])
valid_rows = normalize_split(raw["validation"])
test_rows  = normalize_split(raw["test"])
print(f"[RAW] train={len(train_rows)} valid={len(valid_rows)} test={len(test_rows)}")

# ---- Length filter (whitespace tokens) ----
def within_len(row): 
    return (len(row["src"].split()) <= MAX_LEN) and (len(row["tgt"].split()) <= MAX_LEN)

train_rows = [r for r in train_rows if within_len(r)]
valid_rows = [r for r in valid_rows if within_len(r)]
test_rows  = [r for r in test_rows  if within_len(r)]
print(f"[LEN≤{MAX_LEN}] train={len(train_rows)} valid={len(valid_rows)} test={len(test_rows)}")

# ---- Deterministic shuffle ----
random.Random(SEED).shuffle(train_rows)
random.Random(SEED+1).shuffle(valid_rows)
random.Random(SEED+2).shuffle(test_rows)

# ---- Subsample safely ----
train_rows = train_rows[:TRN_N]
valid_rows = valid_rows[:VAL_N]
test_rows  = test_rows[:TST_N]
print(f"[SUBSET] train={len(train_rows)} valid={len(valid_rows)} test={len(test_rows)}")

# ---- Save JSONL ----
def write_jsonl(rows: List[Dict[str, str]], path: Path):
    with path.open("w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

train_path = ARTIFACTS / f"train.{LANGPAIR}.jsonl"
valid_path = ARTIFACTS / f"valid.{LANGPAIR}.jsonl"
test_path  = ARTIFACTS / f"test.{LANGPAIR}.jsonl"

write_jsonl(train_rows, train_path)
write_jsonl(valid_rows, valid_path)
write_jsonl(test_rows,  test_path)

print(f"[SAVE] {train_path}")
print(f"[SAVE] {valid_path}")
print(f"[SAVE] {test_path}")

# ---- Preview ----
def preview(rows, k=3):
    print("-" * 80)
    for i, r in enumerate(rows[:k], 1):
        print(f"[{i}] SRC({SRC_LANG}): {r['src']}")
        print(f"    TGT({TGT_LANG}): {r['tgt']}\n")

print("\n[PREVIEW] TRAIN")
preview(train_rows)
print("[PREVIEW] VALID")
preview(valid_rows)
print("[PREVIEW] TEST")
preview(test_rows)

# ---- Basic stats ----
def avg_len(rows, key):
    return sum(len(r[key].split()) for r in rows) / max(1, len(rows))

print("-" * 80)
print(f"[STATS] train avg len src={avg_len(train_rows,'src'):.1f} tgt={avg_len(train_rows,'tgt'):.1f}")
print(f"[STATS] valid avg len src={avg_len(valid_rows,'src'):.1f} tgt={avg_len(valid_rows,'tgt'):.1f}")
print(f"[STATS] test  avg len src={avg_len(test_rows,'src'):.1f} tgt={avg_len(test_rows,'tgt'):.1f}")


[CONFIG] {'seed': 42, 'base_dir': 'C:\\Users\\murth\\Desktop\\nlpSession02\\codeBase', 'proj_dir': 'C:\\Users\\murth\\Desktop\\nlpSession02\\codeBase\\translation', 'dataset': 'iwslt2017', 'lang_pair': 'de-en', 'train_size': 5000, 'valid_size': 1000, 'test_size': 1000, 'bpe_vocab': 8000, 'max_len': 40, 'lstm_hidden': 256, 'lstm_layers': 2, 'epochs': 3}
[LOAD] datasets.load_dataset('iwslt2017', 'iwslt2017-de-en')


Generating train split:   0%|          | 0/206112 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/8079 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/888 [00:00<?, ? examples/s]

[RAW] train=206112 valid=888 test=8079
[LEN≤40] train=194588 valid=813 test=7718
[SUBSET] train=5000 valid=813 test=1000
[SAVE] C:\Users\murth\Desktop\nlpSession02\codeBase\translation\artifacts\train.de-en.jsonl
[SAVE] C:\Users\murth\Desktop\nlpSession02\codeBase\translation\artifacts\valid.de-en.jsonl
[SAVE] C:\Users\murth\Desktop\nlpSession02\codeBase\translation\artifacts\test.de-en.jsonl

[PREVIEW] TRAIN
--------------------------------------------------------------------------------
[1] SRC(de): Jetzt ... lege ich den Laser hierhin und - jetzt, wenn ich die Karten im Laser ausgebe weiß ich, wo sie sind, aber - ja?
    TGT(en): Now, I put the laser here, and -- now, when I deal the cards in the laser, I know where they are but -- yes?

[2] SRC(de): Zusammen mit meinem Freund Marco beschloss ich, dorthinzugehen und nachzusehen, wer die echten Palästinenser und wer die echten Israelis sind.
    TGT(en): So with my friend Marco, we decided to go there and see who are the real Palesti

### 🔎 Block 3 — Peek & Sanity Checks

**Goal.** Quickly sanity-check the prepared splits before building models.

**What happens here:**
- Load `train/valid/test .jsonl` from `artifacts/`.
- Print 5 random examples (src→tgt).
- Show length stats (min/mean/95th/max) for both src and tgt.
- (Optional) Plot simple token-length histograms (matplotlib).

> If anything looks off (very long sentences, empty lines, wrong language), go back to Block 2 and adjust `MAX_LEN` or sizes.


In [4]:
# --- Block 3 Code: Peek & Sanity Checks ---
from pathlib import Path
import json, random, statistics as stats
from collections import Counter
import math

import matplotlib.pyplot as plt  # used only for optional histogram

BASE_DIR = Path(r"C:\Users\murth\Desktop\nlpSession02\codeBase")
PROJ_DIR = BASE_DIR / "translation"
ARTIFACTS = PROJ_DIR / "artifacts"

cfg_path = PROJ_DIR / "mt_config.json"
with open(cfg_path, "r", encoding="utf-8") as f:
    CONFIG = json.load(f)
SEED     = int(CONFIG.get("seed", 42))
LANGPAIR = CONFIG.get("lang_pair", "de-en")
SRC_LANG, TGT_LANG = LANGPAIR.split("-")
random.seed(SEED)

def read_jsonl(path: Path):
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

train_path = ARTIFACTS / f"train.{LANGPAIR}.jsonl"
valid_path = ARTIFACTS / f"valid.{LANGPAIR}.jsonl"
test_path  = ARTIFACTS / f"test.{LANGPAIR}.jsonl"

assert train_path.exists() and valid_path.exists() and test_path.exists(), "Missing split files from Block 2."

train_rows = read_jsonl(train_path)
valid_rows = read_jsonl(valid_path)
test_rows  = read_jsonl(test_path)

print(f"[LOAD] train={len(train_rows)} valid={len(valid_rows)} test={len(test_rows)}")

def tok_len(s: str) -> int:
    return len(s.split())

def length_stats(rows, key):
    lens = [tok_len(r[key]) for r in rows]
    if not lens:
        return {"min": 0, "mean": 0, "p95": 0, "max": 0}
    lens_sorted = sorted(lens)
    p95 = lens_sorted[min(len(lens_sorted)-1, math.floor(0.95*(len(lens_sorted)-1)))]
    return {
        "min": min(lens),
        "mean": sum(lens)/len(lens),
        "p95": p95,
        "max": max(lens),
    }

# --- Print 5 random examples from each split ---
def preview(rows, k=5, title="PREVIEW"):
    print("\n" + "="*80)
    print(f"[{title}] {k} random examples")
    print("="*80)
    for i, r in enumerate(random.sample(rows, min(k, len(rows))), 1):
        print(f"[{i}] SRC({SRC_LANG}): {r['src']}")
        print(f"    TGT({TGT_LANG}): {r['tgt']}\n")

preview(train_rows, title="TRAIN")
preview(valid_rows, title="VALID")
preview(test_rows,  title="TEST")

# --- Show length stats ---
def show_stats(name, rows):
    s_src = length_stats(rows, "src")
    s_tgt = length_stats(rows, "tgt")
    print("-"*80)
    print(f"[{name}] src({SRC_LANG}) len  min={s_src['min']:>2}  mean={s_src['mean']:.1f}  p95={s_src['p95']:>2}  max={s_src['max']:>2}")
    print(f"[{name}] tgt({TGT_LANG}) len  min={s_tgt['min']:>2}  mean={s_tgt['mean']:.1f}  p95={s_tgt['p95']:>2}  max={s_tgt['max']:>2}")

print("\n[LENGTH STATS]")
show_stats("TRAIN", train_rows)
show_stats("VALID", valid_rows)
show_stats("TEST",  test_rows)

# --- Optional: simple histograms of token lengths (comment out if not needed) ---
def plot_len_hist(rows, key, title):
    lens = [tok_len(r[key]) for r in rows]
    plt.figure()
    plt.hist(lens, bins=30)
    plt.title(title)
    plt.xlabel("Token length")
    plt.ylabel("Count")
    plt.show()

# Uncomment to visualize
# plot_len_hist(train_rows, "src", f"Train {SRC_LANG} length histogram")
# plot_len_hist(train_rows, "tgt", f"Train {TGT_LANG} length histogram")


[LOAD] train=5000 valid=813 test=1000

[TRAIN] 5 random examples
[1] SRC(de): Zum nächsten Teil dieser Geschichtsstunde, hier ist ein hübsches Bild der industriellen Revolution in Großbritannien.
    TGT(en): Next part of the cold history lesson, the lovely picture of the British Industrial Revolution.

[2] SRC(de): Wir übernehmen sie von anderen Leuten. Männer hauptsächlich von ihren Vätern. Und Frauen von ihren Müttern.
    TGT(en): They're sucked in from other people; chiefly, if you're a man, your father, and if you're a woman, your mother.

[3] SRC(de): Sie wurde hergestellt, indem künstlicher Sandstein Schicht für Schicht in 5 bis 10 Millimeter dicken Schichten abgelagert wurde – die Struktur wird langsam aufgebaut.
    TGT(en): And this was built by depositing artificial sandstone layer upon layer in layers of about five millimeters to 10 mm in thickness -- slowly growing this structure.

[4] SRC(de): Dann kommen wir zur Zivilisation – über dem kleinen Fernseher mit der Pistole.

### 🧯 Block 4 — Pre-Neural Baseline (word-for-word SMT-like)

**Goal.** Build a tiny *pre-neural* baseline to show why older methods struggle.  
We’ll learn a **frequency dictionary** `src_token → most_frequent_tgt_token` from the **train** set and translate the **test** set word-by-word.

**Method (simple but illustrative):**
- Tokenize by whitespace, lowercase.  
- For each train pair `(src_sentence, tgt_sentence)`, **co-occurrence count** every `(src_token, tgt_token)` in that pair (bag-of-words, no alignment).  
- For each `src_token`, pick the **most frequent** `tgt_token`.  
- Decode test by mapping each `src_token` → dictionary (fallback: copy source token).  
- Save predictions to `artifacts/baseline.de-en.jsonl` for later BLEU/BERTScore.

> This baseline ignores alignment, reordering, morphology, etc. Expect low BLEU — that’s the point. We’ll contrast it with LSTM and Transformer later.


In [5]:
# --- Block 4 Code: Pre-Neural Baseline (word-for-word SMT-like) ---
from pathlib import Path
import json, random
from collections import defaultdict, Counter

BASE_DIR = Path(r"C:\Users\murth\Desktop\nlpSession02\codeBase")
PROJ_DIR = BASE_DIR / "translation"
ARTIFACTS = PROJ_DIR / "artifacts"

cfg_path = PROJ_DIR / "mt_config.json"
with open(cfg_path, "r", encoding="utf-8") as f:
    CONFIG = json.load(f)
SEED     = int(CONFIG.get("seed", 42))
LANGPAIR = CONFIG.get("lang_pair", "de-en")
SRC_LANG, TGT_LANG = LANGPAIR.split("-")
random.seed(SEED)

def read_jsonl(path: Path):
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

train_path = ARTIFACTS / f"train.{LANGPAIR}.jsonl"
valid_path = ARTIFACTS / f"valid.{LANGPAIR}.jsonl"
test_path  = ARTIFACTS / f"test.{LANGPAIR}.jsonl"
assert train_path.exists() and valid_path.exists() and test_path.exists(), "Missing splits (run Blocks 2–3)."

train_rows = read_jsonl(train_path)
valid_rows = read_jsonl(valid_path)
test_rows  = read_jsonl(test_path)
print(f"[LOAD] train={len(train_rows)} valid={len(valid_rows)} test={len(test_rows)}")

# --- Tiny tokenizer: lowercase, split on whitespace (keeps punctuation as tokens) ---
def tok(s: str):
    return s.lower().strip().split()

# --- Learn co-occurrence dictionary from TRAIN ---
pair_counts = defaultdict(Counter)  # src_token -> Counter({tgt_token: count})
for ex in train_rows:
    src_tokens = tok(ex["src"])
    tgt_tokens = tok(ex["tgt"])
    if not src_tokens or not tgt_tokens:
        continue
    # Bag-of-words co-occurrence (very naive, but enough to illustrate)
    tgt_set = set(tgt_tokens)  # avoid overweighting repeated tgt tokens in same sentence
    for s in set(src_tokens):  # also unique per sentence to reduce spam
        pair_counts[s].update(tgt_set)

# Build final lexicon: pick the single most frequent target for each source token
lexicon = {}
for s, ctr in pair_counts.items():
    tgt, cnt = ctr.most_common(1)[0]
    lexicon[s] = tgt

print(f"[LEXICON] learned {len(lexicon)} source tokens → target mappings")
# Show 15 random mappings
for i, (s, t) in enumerate(random.sample(list(lexicon.items()), min(15, len(lexicon))), 1):
    print(f"  {i:>2}. {s!r} → {t!r}")

# --- Translate TEST word-by-word with fallback to copying the src token ---
def translate_baseline(src_sentence: str) -> str:
    out = []
    for s in tok(src_sentence):
        out.append(lexicon.get(s, s))  # fallback = copy source token
    return " ".join(out)

# Produce predictions on TEST
pred_rows = []
for ex in test_rows:
    hyp = translate_baseline(ex["src"])
    pred_rows.append({"src": ex["src"], "ref": ex["tgt"], "hyp": hyp})

# Save for later metrics
out_path = ARTIFACTS / f"baseline.{LANGPAIR}.jsonl"
with out_path.open("w", encoding="utf-8") as f:
    for r in pred_rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print(f"[SAVE] {out_path}  ({len(pred_rows)} lines)")

# Preview a few translations
print("\n[PREVIEW] Baseline translations (5 examples)")
for r in random.sample(pred_rows, min(5, len(pred_rows))):
    print("-"*80)
    print("SRC:", r["src"])
    print("REF:", r["ref"])
    print("HYP:", r["hyp"])


[LOAD] train=5000 valid=813 test=1000
[LEXICON] learned 16422 source tokens → target mappings
   1. 'möbeln,' → 'winning'
   2. 'passieren' → 'the'
   3. 'ergründen,' → 'illuminated'
   4. 'unterschiedliche,' → 'conflicts'
   5. 'machen;' → 'well,'
   6. 'euch.' → 'one'
   7. 'u-bahn' → 'to'
   8. 'widerspruch' → 'to'
   9. 'geographic,' → "i'm"
  10. 'sie.' → 'you'
  11. 'stieg' → 'to'
  12. 'anderweitig' → 'that'
  13. 'andernfalls' → 'when'
  14. 'lungenkapazität' → 'white'
  15. 'bahamas' → 'bahamas'
[SAVE] C:\Users\murth\Desktop\nlpSession02\codeBase\translation\artifacts\baseline.de-en.jsonl  (1000 lines)

[PREVIEW] Baseline translations (5 examples)
--------------------------------------------------------------------------------
SRC: Und Dr. P. trug immer sehr bunte Fliegen und war für die Arbeit mit Kindern einfach wie geschaffen.
REF: And Dr. P always wore really colorful bow ties  and had the very perfect disposition to work with children.
HYP: and dr. p. speed the very bunte

### 📊 Block 5 — Baseline Evaluation (BLEU + BERTScore)

**Goal.** Evaluate the pre-neural baseline on the **test** split using **SacreBLEU** (form overlap) and **BERTScore** (semantic similarity).

**What happens here:**
- Load `artifacts/baseline.de-en.jsonl`.
- Compute **BLEU** with `sacrebleu.corpus_bleu`.
- Compute **BERTScore** (P/R/F1) with `bert_score.score(lang="en")`.
- Print a compact summary and save `artifacts/metrics.baseline.json`.


In [6]:
# --- Block 5 Code: Baseline Evaluation (BLEU + BERTScore) ---
from pathlib import Path
import json, statistics as stats
import sacrebleu
from bert_score import score as bertscore

BASE_DIR = Path(r"C:\Users\murth\Desktop\nlpSession02\codeBase")
PROJ_DIR = BASE_DIR / "translation"
ARTIFACTS = PROJ_DIR / "artifacts"

cfg_path = PROJ_DIR / "mt_config.json"
with open(cfg_path, "r", encoding="utf-8") as f:
    CONFIG = json.load(f)
LANGPAIR = CONFIG.get("lang_pair", "de-en")

pred_path = ARTIFACTS / f"baseline.{LANGPAIR}.jsonl"
assert pred_path.exists(), f"Predictions not found: {pred_path}. Run Block 4."

# Load predictions
rows = []
with pred_path.open("r", encoding="utf-8") as f:
    for line in f:
        rows.append(json.loads(line))

refs = [r["ref"] for r in rows]
hyps = [r["hyp"] for r in rows]

# --- SacreBLEU (uses detokenized strings) ---
bleu = sacrebleu.corpus_bleu(hyps, [refs])  # references must be list-of-lists
bleu_score = bleu.score

# --- BERTScore (semantic) ---
# Note: lang='en' because target is English. Returns per-sentence P/R/F1 tensors.
P, R, F1 = bertscore(hyps, refs, lang="en", rescale_with_baseline=True)
bert_p = float(P.mean())
bert_r = float(R.mean())
bert_f1 = float(F1.mean())

print("\n=== Baseline Metrics ===")
print(f"BLEU:          {bleu_score:.2f}")
print(f"BERTScore-P:   {bert_p:.4f}")
print(f"BERTScore-R:   {bert_r:.4f}")
print(f"BERTScore-F1:  {bert_f1:.4f}")

# Save metrics
metrics = {
    "model": "baseline_word_dictionary",
    "langpair": LANGPAIR,
    "bleu": bleu_score,
    "bertscore": {"precision": bert_p, "recall": bert_r, "f1": bert_f1},
    "n_samples": len(rows),
}
out_path = ARTIFACTS / f"metrics.baseline.{LANGPAIR}.json"
with out_path.open("w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)
print(f"[SAVE] {out_path}")


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

c:\Users\murth\Desktop\nlpSession02\codeBase\.venv\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\murth\.cache\huggingface\hub\models--roberta-large. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



=== Baseline Metrics ===
BLEU:          1.90
BERTScore-P:   -0.1586
BERTScore-R:   0.0953
BERTScore-F1:  -0.0344
[SAVE] C:\Users\murth\Desktop\nlpSession02\codeBase\translation\artifacts\metrics.baseline.de-en.json


### 🔤 Block 6 — Train Subword Tokenizer (SentencePiece BPE, joint vocab)

**Goal.** Learn a small **joint** BPE tokenizer (German+English) so both LSTM and Transformer demos can reuse consistent subwords.

**What happens here:**
- Read `train.de-en.jsonl`.
- Create a temporary training text combining **src** and **tgt** lines.
- Train **SentencePiece BPE** with vocab size = `bpe_vocab` (from `mt_config.json`, default **8000**).
- Save to `artifacts/spm_bpe_{vocab}.model` and `.vocab`.
- Provide quick encode/decode smoke tests.

> Joint vocab keeps things simple for class and reduces OOVs. We’ll cap sequence lengths later when building PyTorch datasets.


In [7]:
# --- Block 6 Code: Train SentencePiece BPE (joint de+en) ---
from pathlib import Path
import json, os, random
import sentencepiece as spm

BASE_DIR = Path(r"C:\Users\murth\Desktop\nlpSession02\codeBase")
PROJ_DIR = BASE_DIR / "translation"
ARTIFACTS = PROJ_DIR / "artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)

# Load config
cfg_path = PROJ_DIR / "mt_config.json"
with open(cfg_path, "r", encoding="utf-8") as f:
    CONFIG = json.load(f)
SEED      = int(CONFIG.get("seed", 42))
LANGPAIR  = CONFIG.get("lang_pair", "de-en")
BPE_VOCAB = int(CONFIG.get("bpe_vocab", 8000))

train_path = ARTIFACTS / f"train.{LANGPAIR}.jsonl"
assert train_path.exists(), f"Missing train split: {train_path}. Run Block 2."

# Prepare a temporary corpus file for SentencePiece (joint src+tgt)
tmp_corpus = ARTIFACTS / f"spm_corpus.{LANGPAIR}.txt"
random.seed(SEED)

def iter_jsonl(path):
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            yield json.loads(line)

# Write src+tgt lines, shuffled lightly for variety
rows = list(iter_jsonl(train_path))
random.shuffle(rows)
with tmp_corpus.open("w", encoding="utf-8") as f:
    for r in rows:
        src = r["src"].strip()
        tgt = r["tgt"].strip()
        if src: f.write(src + "\n")
        if tgt: f.write(tgt + "\n")

print(f"[CORPUS] {tmp_corpus}  lines≈{len(rows)*2}")

# Train SentencePiece BPE
spm_prefix = ARTIFACTS / f"spm_bpe_{BPE_VOCAB}"
cmd = (
    f"--input={tmp_corpus} "
    f"--model_prefix={spm_prefix} "
    f"--vocab_size={BPE_VOCAB} "
    f"--model_type=bpe "
    f"--character_coverage=1.0 "
    f"--bos_id=1 --eos_id=2 --unk_id=0 --pad_id=3 "
    f"--input_sentence_size=200000 --shuffle_input_sentence=true "
)
print("[SENTENCEPIECE TRAIN]", cmd)
spm.SentencePieceTrainer.Train(cmd)

model_path = Path(f"{spm_prefix}.model")
vocab_path = Path(f"{spm_prefix}.vocab")
assert model_path.exists() and vocab_path.exists(), "SentencePiece training failed."

print(f"[OK] Saved tokenizer:\n  {model_path}\n  {vocab_path}")

# Quick smoke test: encode/decode a sample pair
sp = spm.SentencePieceProcessor(model_file=str(model_path))

sample_src = rows[0]["src"]
sample_tgt = rows[0]["tgt"]
ids_src = sp.encode(sample_src, out_type=int)
ids_tgt = sp.encode(sample_tgt, out_type=int)
print("\n[SMOKE TEST]")
print("SRC:", sample_src)
print("IDS:", ids_src[:30], "… (len:", len(ids_src), ")")
print("DEC:", sp.decode(ids_src))

print("\nTGT:", sample_tgt)
print("IDS:", ids_tgt[:30], "… (len:", len(ids_tgt), ")")
print("DEC:", sp.decode(ids_tgt))

# (Optional) Clean the temporary corpus if you like:
# tmp_corpus.unlink(missing_ok=True)


[CORPUS] C:\Users\murth\Desktop\nlpSession02\codeBase\translation\artifacts\spm_corpus.de-en.txt  lines≈10000
[SENTENCEPIECE TRAIN] --input=C:\Users\murth\Desktop\nlpSession02\codeBase\translation\artifacts\spm_corpus.de-en.txt --model_prefix=C:\Users\murth\Desktop\nlpSession02\codeBase\translation\artifacts\spm_bpe_8000 --vocab_size=8000 --model_type=bpe --character_coverage=1.0 --bos_id=1 --eos_id=2 --unk_id=0 --pad_id=3 --input_sentence_size=200000 --shuffle_input_sentence=true 
[OK] Saved tokenizer:
  C:\Users\murth\Desktop\nlpSession02\codeBase\translation\artifacts\spm_bpe_8000.model
  C:\Users\murth\Desktop\nlpSession02\codeBase\translation\artifacts\spm_bpe_8000.vocab

[SMOKE TEST]
SRC: Sie hat ein wenig mit der letzten zu tun, aber sie ist viel erotischer.
IDS: [150, 264, 52, 1976, 208, 94, 1527, 104, 797, 7913, 407, 151, 96, 706, 177, 114, 1556, 7916] … (len: 18 )
DEC: Sie hat ein wenig mit der letzten zu tun, aber sie ist viel erotischer.

TGT: So, it's sort of related, but 

### 📚 Block 7 — Seq2Seq Dataset & PyTorch Dataloaders (BPE, BOS/EOS, Padding)

**Goal.** Turn the JSONL splits into **tokenized tensors** for the LSTM model:
- Load the SentencePiece BPE model from Block 6.
- Encode **src (de)** and **tgt (en)** with a **joint vocab**.
- Add special tokens: `BOS=1`, `EOS=2`, `UNK=0`, `PAD=3` (as configured in Block 6).
- Truncate to `max_len` (from `mt_config.json`), create:
  - `tgt_in`  (starts with BOS, no final EOS) → teacher forcing input
  - `tgt_out` (no initial BOS, ends with EOS) → next-token targets
- Build **collate_fn** to pad batches dynamically.
- Create **DataLoaders** for train/valid/test and print sample shapes.

> These loaders feed directly into the **Block 8** LSTM+Attention trainer.


In [8]:
# --- Block 2 Code (Fixed): Data Load & Split (IWSLT2017 de→en) ---
from pathlib import Path
import json, random
from typing import List, Dict
import datasets

# ---- Paths & config ----
BASE_DIR = Path(r"C:\Users\murth\Desktop\nlpSession02\codeBase")
PROJ_DIR = BASE_DIR / "translation"
ARTIFACTS = PROJ_DIR / "artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)

cfg_path = PROJ_DIR / "mt_config.json"
assert cfg_path.exists(), f"Config not found: {cfg_path}. Run Block 1 first."
with open(cfg_path, "r", encoding="utf-8") as f:
    CONFIG = json.load(f)

SEED     = int(CONFIG.get("seed", 42))
MAX_LEN  = int(CONFIG.get("max_len", 40))
TRN_N    = int(CONFIG.get("train_size", 5000))
VAL_N    = int(CONFIG.get("valid_size", 1000))
TST_N    = int(CONFIG.get("test_size", 1000))
LANGPAIR = CONFIG.get("lang_pair", "de-en")
SRC_LANG, TGT_LANG = LANGPAIR.split("-")
random.seed(SEED)

print("[CONFIG]", CONFIG)

# ---- Load dataset (correct builder config) ----
print("[LOAD] datasets.load_dataset('iwslt2017', 'iwslt2017-de-en')")
raw = datasets.load_dataset("iwslt2017", "iwslt2017-de-en")

# keys: 'train', 'validation', 'test'; texts under ex['translation']['de'/'en']
def normalize_split(ds: datasets.Dataset) -> List[Dict[str, str]]:
    rows = []
    for ex in ds:
        tr = ex.get("translation", {})
        src = (tr.get(SRC_LANG) or "").strip()
        tgt = (tr.get(TGT_LANG) or "").strip()
        if src and tgt:
            rows.append({"src": src, "tgt": tgt})
    return rows

train_rows = normalize_split(raw["train"])
valid_rows = normalize_split(raw["validation"])
test_rows  = normalize_split(raw["test"])
print(f"[RAW] train={len(train_rows)} valid={len(valid_rows)} test={len(test_rows)}")

# ---- Length filter (whitespace tokens) ----
def within_len(row): 
    return (len(row["src"].split()) <= MAX_LEN) and (len(row["tgt"].split()) <= MAX_LEN)

train_rows = [r for r in train_rows if within_len(r)]
valid_rows = [r for r in valid_rows if within_len(r)]
test_rows  = [r for r in test_rows  if within_len(r)]
print(f"[LEN≤{MAX_LEN}] train={len(train_rows)} valid={len(valid_rows)} test={len(test_rows)}")

# ---- Deterministic shuffle ----
random.Random(SEED).shuffle(train_rows)
random.Random(SEED+1).shuffle(valid_rows)
random.Random(SEED+2).shuffle(test_rows)

# ---- Subsample safely ----
train_rows = train_rows[:TRN_N]
valid_rows = valid_rows[:VAL_N]
test_rows  = test_rows[:TST_N]
print(f"[SUBSET] train={len(train_rows)} valid={len(valid_rows)} test={len(test_rows)}")

# ---- Save JSONL ----
def write_jsonl(rows: List[Dict[str, str]], path: Path):
    with path.open("w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

train_path = ARTIFACTS / f"train.{LANGPAIR}.jsonl"
valid_path = ARTIFACTS / f"valid.{LANGPAIR}.jsonl"
test_path  = ARTIFACTS / f"test.{LANGPAIR}.jsonl"

write_jsonl(train_rows, train_path)
write_jsonl(valid_rows, valid_path)
write_jsonl(test_rows,  test_path)

print(f"[SAVE] {train_path}")
print(f"[SAVE] {valid_path}")
print(f"[SAVE] {test_path}")

# ---- Preview ----
def preview(rows, k=3):
    print("-" * 80)
    for i, r in enumerate(rows[:k], 1):
        print(f"[{i}] SRC({SRC_LANG}): {r['src']}")
        print(f"    TGT({TGT_LANG}): {r['tgt']}\n")

print("\n[PREVIEW] TRAIN")
preview(train_rows)
print("[PREVIEW] VALID")
preview(valid_rows)
print("[PREVIEW] TEST")
preview(test_rows)

# ---- Basic stats ----
def avg_len(rows, key):
    return sum(len(r[key].split()) for r in rows) / max(1, len(rows))

print("-" * 80)
print(f"[STATS] train avg len src={avg_len(train_rows,'src'):.1f} tgt={avg_len(train_rows,'tgt'):.1f}")
print(f"[STATS] valid avg len src={avg_len(valid_rows,'src'):.1f} tgt={avg_len(valid_rows,'tgt'):.1f}")
print(f"[STATS] test  avg len src={avg_len(test_rows,'src'):.1f} tgt={avg_len(test_rows,'tgt'):.1f}")


[CONFIG] {'seed': 42, 'base_dir': 'C:\\Users\\murth\\Desktop\\nlpSession02\\codeBase', 'proj_dir': 'C:\\Users\\murth\\Desktop\\nlpSession02\\codeBase\\translation', 'dataset': 'iwslt2017', 'lang_pair': 'de-en', 'train_size': 5000, 'valid_size': 1000, 'test_size': 1000, 'bpe_vocab': 8000, 'max_len': 40, 'lstm_hidden': 256, 'lstm_layers': 2, 'epochs': 3}
[LOAD] datasets.load_dataset('iwslt2017', 'iwslt2017-de-en')


c:\Users\murth\Desktop\nlpSession02\codeBase\.venv\lib\site-packages\datasets\load.py:1486: FutureWarning: The repository for iwslt2017 contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/iwslt2017
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


[RAW] train=206112 valid=888 test=8079
[LEN≤40] train=194588 valid=813 test=7718
[SUBSET] train=5000 valid=813 test=1000
[SAVE] C:\Users\murth\Desktop\nlpSession02\codeBase\translation\artifacts\train.de-en.jsonl
[SAVE] C:\Users\murth\Desktop\nlpSession02\codeBase\translation\artifacts\valid.de-en.jsonl
[SAVE] C:\Users\murth\Desktop\nlpSession02\codeBase\translation\artifacts\test.de-en.jsonl

[PREVIEW] TRAIN
--------------------------------------------------------------------------------
[1] SRC(de): Jetzt ... lege ich den Laser hierhin und - jetzt, wenn ich die Karten im Laser ausgebe weiß ich, wo sie sind, aber - ja?
    TGT(en): Now, I put the laser here, and -- now, when I deal the cards in the laser, I know where they are but -- yes?

[2] SRC(de): Zusammen mit meinem Freund Marco beschloss ich, dorthinzugehen und nachzusehen, wer die echten Palästinenser und wer die echten Israelis sind.
    TGT(en): So with my friend Marco, we decided to go there and see who are the real Palesti

### 🧠 Block 8 — LSTM + Additive Attention: Model & Training (CPU-friendly)

**Goal.** Train a tiny **Seq2Seq (2×256 LSTM) + Bahdanau attention** on our BPE data.

**Design choices**
- **Shared embedding** (joint vocab from SentencePiece) for src/tgt to keep it small.
- **Encoder**: 2-layer biLSTM → concat forward/backward → project to decoder dim.
- **Decoder**: 2-layer uniLSTM + additive attention (Bahdanau) each step.
- **Loss**: Cross-entropy on `tgt_out` (ignore `PAD=3`). Teacher-forcing during training.
- **Optim**: AdamW (lr=3e-4), gradient clip=1.0, 3 epochs (increase to 5 if time permits).
- **Artifacts**: best checkpoint → `artifacts/lstm_attn_best.pt` + training log.

> This is a teaching model (clean & readable). It’s not optimized for speed; still fine on CPU for our tiny splits. We’ll compare with Transformer later.


In [9]:
# --- Block 8 Code: LSTM + Additive Attention model & training ---
from pathlib import Path
import json, math, time
from typing import Tuple, Dict

import torch
import torch.nn as nn
from torch.nn.utils import clip_grad_norm_
import sentencepiece as spm

# ----------------- Paths & Config -----------------
BASE_DIR = Path(r"C:\Users\murth\Desktop\nlpSession02\codeBase")
PROJ_DIR = BASE_DIR / "translation"
ARTIFACTS = PROJ_DIR / "artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)

cfg_path = PROJ_DIR / "mt_config.json"
with open(cfg_path, "r", encoding="utf-8") as f:
    CONFIG = json.load(f)

SEED        = int(CONFIG.get("seed", 42))
MAX_LEN     = int(CONFIG.get("max_len", 40))
BPE_VOCAB   = int(CONFIG.get("bpe_vocab", 8000))
HIDDEN      = int(CONFIG.get("lstm_hidden", 256))
LAYERS      = int(CONFIG.get("lstm_layers", 2))
EPOCHS      = int(CONFIG.get("epochs", 3))
BATCH_SIZE  = int(CONFIG.get("batch_size", 64))
DROPOUT     = float(CONFIG.get("dropout", 0.2))
LR          = float(CONFIG.get("lr", 3e-4))
CLIP_NORM   = float(CONFIG.get("clip_norm", 1.0))
LANGPAIR    = CONFIG.get("lang_pair", "de-en")

# Special IDs
ID_UNK = 0
ID_BOS = 1
ID_EOS = 2
ID_PAD = 3

# ----------------- Reuse dataloaders from Block 7 -----------------
# Expect that user ran Block 7; import-friendly approach:
import importlib, sys
# we will quickly rebuild loaders in-place using Block 7 logic to avoid cross-cell state issues
from torch.utils.data import DataLoader, Dataset

spm_model_path = ARTIFACTS / f"spm_bpe_{BPE_VOCAB}.model"
assert spm_model_path.exists(), "SentencePiece model missing. Run Block 6."
sp = spm.SentencePieceProcessor(model_file=str(spm_model_path))
VOCAB_SIZE = sp.get_piece_size()

def read_jsonl(path: Path):
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

def encode_with_bpe(text: str, max_len: int):
    ids = sp.encode(text, out_type=int)
    return ids[:max_len]

def make_tgt_in_out(tgt_ids):
    tgt_in  = [ID_BOS] + tgt_ids
    tgt_out = tgt_ids + [ID_EOS]
    L = min(len(tgt_in), len(tgt_out))
    return tgt_in[:L], tgt_out[:L]

class Seq2SeqJsonl(Dataset):
    def __init__(self, path: Path, max_len: int):
        self.rows = read_jsonl(path)
        self.max_len = max_len
    def __len__(self): return len(self.rows)
    def __getitem__(self, idx):
        r = self.rows[idx]
        src_ids = encode_with_bpe(r["src"], self.max_len)
        tgt_ids = encode_with_bpe(r["tgt"], self.max_len)
        tgt_in, tgt_out = make_tgt_in_out(tgt_ids)
        return {
            "src_ids": torch.tensor(src_ids, dtype=torch.long),
            "tgt_in":  torch.tensor(tgt_in,  dtype=torch.long),
            "tgt_out": torch.tensor(tgt_out, dtype=torch.long),
        }

def collate_batch(batch):
    bs = len(batch)
    max_src = max(len(b["src_ids"]) for b in batch)
    max_in  = max(len(b["tgt_in"])  for b in batch)
    max_out = max(len(b["tgt_out"]) for b in batch)

    src_pad = torch.full((bs, max_src), ID_PAD, dtype=torch.long)
    in_pad  = torch.full((bs, max_in),  ID_PAD, dtype=torch.long)
    out_pad = torch.full((bs, max_out), ID_PAD, dtype=torch.long)

    for i, b in enumerate(batch):
        s, ti, to = b["src_ids"], b["tgt_in"], b["tgt_out"]
        src_pad[i, :len(s)] = s
        in_pad[i,  :len(ti)] = ti
        out_pad[i, :len(to)] = to

    src_mask = (src_pad != ID_PAD).long()
    tgt_mask = (in_pad  != ID_PAD).long()

    return {
        "src_ids": src_pad,   # [B,Ss]
        "src_mask": src_mask, # [B,Ss]
        "tgt_in": in_pad,     # [B,St]
        "tgt_out": out_pad,   # [B,St]
        "tgt_mask": tgt_mask, # [B,St]
    }

train_path = ARTIFACTS / f"train.{LANGPAIR}.jsonl"
valid_path = ARTIFACTS / f"valid.{LANGPAIR}.jsonl"
test_path  = ARTIFACTS / f"test.{LANGPAIR}.jsonl"
assert train_path.exists(), "Run Block 2."
assert valid_path.exists() and test_path.exists(), "Run Block 2."

train_loader = DataLoader(Seq2SeqJsonl(train_path, MAX_LEN), batch_size=BATCH_SIZE, shuffle=True,  collate_fn=collate_batch, num_workers=0)
valid_loader = DataLoader(Seq2SeqJsonl(valid_path, MAX_LEN), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_batch, num_workers=0)

# ----------------- Model components -----------------
torch.manual_seed(SEED)
device = torch.device("cpu")  # keep CPU for classroom consistency

class Encoder(nn.Module):
    def __init__(self, vocab_size: int, emb_dim: int, hidden: int, layers: int, dropout: float):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=ID_PAD)
        self.rnn = nn.LSTM(emb_dim, hidden, num_layers=layers, dropout=dropout if layers > 1 else 0.0,
                           bidirectional=True, batch_first=True)
        # Project bi-hidden (2*hidden) to decoder hidden
        self.fc_h = nn.Linear(hidden*2, hidden)
        self.fc_c = nn.Linear(hidden*2, hidden)

    def forward(self, src_ids, src_mask):
        # src_ids: [B,Ss]
        x = self.emb(src_ids)             # [B,Ss,E]
        outputs, (h_n, c_n) = self.rnn(x) # outputs: [B,Ss,2H]
        # Use last time-step from both directions by concatenation per layer is messy;
        # Simpler: take outputs and compute a pooled summary (mask-aware mean) to init decoder.
        mask = src_mask.unsqueeze(-1)     # [B,Ss,1]
        summed = (outputs * mask).sum(1)  # [B,2H]
        lens = mask.sum(1).clamp(min=1)   # [B,1]
        mean_ctx = summed / lens          # [B,2H]
        h0 = torch.tanh(self.fc_h(mean_ctx)).unsqueeze(0).repeat(self.rnn.num_layers//1, 1, 1)  # [L,B,H]
        c0 = torch.tanh(self.fc_c(mean_ctx)).unsqueeze(0).repeat(self.rnn.num_layers//1, 1, 1)  # [L,B,H]
        return outputs, (h0, c0)          # encoder outputs (for attention), and projected state

class AdditiveAttention(nn.Module):
    """Bahdanau attention: score(s_t, h_i) = v^T tanh(W_h h_i + W_s s_t)"""
    def __init__(self, enc_dim: int, dec_dim: int, attn_dim: int):
        super().__init__()
        self.W_h = nn.Linear(enc_dim, attn_dim, bias=False)
        self.W_s = nn.Linear(dec_dim, attn_dim, bias=False)
        self.v   = nn.Linear(attn_dim, 1, bias=False)

    def forward(self, dec_state, enc_outputs, mask):
        # dec_state: [B,H], enc_outputs: [B,Ss,2H] or [B,Ss,Henc], mask: [B,Ss]
        # Project both
        Wh = self.W_h(enc_outputs)                          # [B,Ss,A]
        Ws = self.W_s(dec_state).unsqueeze(1)               # [B,1,A]
        scores = self.v(torch.tanh(Wh + Ws)).squeeze(-1)    # [B,Ss]
        scores = scores.masked_fill(mask == 0, -1e9)        # mask pads
        attn = torch.softmax(scores, dim=-1)                # [B,Ss]
        ctx = torch.bmm(attn.unsqueeze(1), enc_outputs).squeeze(1)  # [B,Henc]
        return ctx, attn                                    # context vector + weights

class Decoder(nn.Module):
    def __init__(self, vocab_size: int, emb_dim: int, hidden: int, layers: int, dropout: float, enc_out_dim: int):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=ID_PAD)
        self.rnn = nn.LSTM(emb_dim + enc_out_dim, hidden, num_layers=layers, dropout=dropout if layers > 1 else 0.0,
                           batch_first=True)
        self.attn = AdditiveAttention(enc_out_dim, hidden, attn_dim=hidden)
        self.fc_out = nn.Linear(hidden + enc_out_dim, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, tgt_in, hidden, cell, enc_outputs, src_mask):
        """
        Teacher-forced decoding over the whole target sequence.
        tgt_in: [B,St] (starts with BOS)
        Returns logits: [B,St,V]
        """
        B, St = tgt_in.size()
        emb = self.dropout(self.emb(tgt_in))                # [B,St,E]
        outputs = []
        h, c = hidden, cell
        # Step-by-step to apply attention at each time
        for t in range(St):
            # Compute context from current decoder state (use top layer hidden)
            dec_state = h[-1]                               # [B,H]
            ctx, _ = self.attn(dec_state, enc_outputs, src_mask)  # [B,Henc]
            rnn_in = torch.cat([emb[:, t, :], ctx], dim=-1).unsqueeze(1)  # [B,1,E+Henc]
            out_t, (h, c) = self.rnn(rnn_in, (h, c))        # out_t: [B,1,H]
            out_t = out_t.squeeze(1)                        # [B,H]
            logits_t = self.fc_out(torch.cat([out_t, ctx], dim=-1))  # [B,V]
            outputs.append(logits_t.unsqueeze(1))
        return torch.cat(outputs, dim=1)  # [B,St,V]

class Seq2Seq(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden, layers, dropout):
        super().__init__()
        self.encoder = Encoder(vocab_size, emb_dim, hidden, layers, dropout)
        # Encoder is biLSTM → enc_out_dim = 2*hidden
        self.decoder = Decoder(vocab_size, emb_dim, hidden, layers, dropout, enc_out_dim=hidden*2)

    def forward(self, src_ids, src_mask, tgt_in):
        enc_outputs, (h0, c0) = self.encoder(src_ids, src_mask)  # enc_outputs: [B,Ss,2H]
        logits = self.decoder(tgt_in, h0, c0, enc_outputs, src_mask)  # [B,St,V]
        return logits

# ----------------- Training helpers -----------------
def sequence_ce_loss(logits, targets, pad_id=ID_PAD):
    """
    logits: [B,St,V], targets: [B,St]
    Ignores PAD positions in targets.
    """
    B, St, V = logits.size()
    loss_fn = nn.CrossEntropyLoss(ignore_index=pad_id)
    return loss_fn(logits.reshape(B*St, V), targets.reshape(B*St))

@torch.no_grad()
def evaluate(model: nn.Module, loader) -> float:
    model.eval()
    total, n = 0.0, 0
    for batch in loader:
        src_ids  = batch["src_ids"].to(device)
        src_mask = batch["src_mask"].to(device)
        tgt_in   = batch["tgt_in"].to(device)
        tgt_out  = batch["tgt_out"].to(device)
        logits = model(src_ids, src_mask, tgt_in)
        loss = sequence_ce_loss(logits, tgt_out, pad_id=ID_PAD)
        total += loss.item()
        n += 1
    return total / max(1, n)

def train(model: nn.Module, train_loader, valid_loader, epochs: int):
    opt = torch.optim.AdamW(model.parameters(), lr=LR)
    best_val = float("inf")
    best_path = ARTIFACTS / "lstm_attn_best.pt"
    history = []

    for ep in range(1, epochs+1):
        model.train()
        t0 = time.time()
        total, n = 0.0, 0
        for batch in train_loader:
            src_ids  = batch["src_ids"].to(device)
            src_mask = batch["src_mask"].to(device)
            tgt_in   = batch["tgt_in"].to(device)
            tgt_out  = batch["tgt_out"].to(device)

            opt.zero_grad()
            logits = model(src_ids, src_mask, tgt_in)    # [B,St,V]
            loss = sequence_ce_loss(logits, tgt_out, pad_id=ID_PAD)
            loss.backward()
            clip_grad_norm_(model.parameters(), CLIP_NORM)
            opt.step()

            total += loss.item()
            n += 1

        train_loss = total / max(1, n)
        val_loss = evaluate(model, valid_loader)
        dt = time.time() - t0
        history.append({"epoch": ep, "train_loss": train_loss, "valid_loss": val_loss, "time_sec": dt})

        print(f"Epoch {ep:02d} | train CE: {train_loss:.4f} | valid CE: {val_loss:.4f} | {dt:.1f}s")

        if val_loss < best_val:
            best_val = val_loss
            torch.save({
                "model_state": model.state_dict(),
                "config": {
                    "vocab_size": VOCAB_SIZE,
                    "emb_dim": HIDDEN,     # we’ll set emb_dim == hidden below
                    "hidden": HIDDEN,
                    "layers": LAYERS,
                    "dropout": DROPOUT,
                    "pad_id": ID_PAD,
                    "langpair": LANGPAIR,
                    "bpe_vocab": BPE_VOCAB,
                }
            }, best_path)
            print(f"[💾] Saved best checkpoint → {best_path} (valid CE={best_val:.4f})")

    # Save training log
    with (ARTIFACTS / "lstm_attn_train_log.json").open("w", encoding="utf-8") as f:
        json.dump(history, f, indent=2)
    print(f"[OK] Training complete. Best valid CE={best_val:.4f}")

# ----------------- Instantiate & Train -----------------
EMB_DIM = HIDDEN  # simple choice: tie embedding dim to hidden size
model = Seq2Seq(VOCAB_SIZE, EMB_DIM, HIDDEN, LAYERS, DROPOUT).to(device)
print(f"[MODEL] vocab={VOCAB_SIZE} emb={EMB_DIM} hidden={HIDDEN} layers={LAYERS} dropout={DROPOUT}")
train(model, train_loader, valid_loader, epochs=EPOCHS)


[MODEL] vocab=8000 emb=256 hidden=256 layers=2 dropout=0.2
Epoch 01 | train CE: 6.9877 | valid CE: 6.3882 | 359.2s
[💾] Saved best checkpoint → C:\Users\murth\Desktop\nlpSession02\codeBase\translation\artifacts\lstm_attn_best.pt (valid CE=6.3882)
Epoch 02 | train CE: 6.1973 | valid CE: 6.1997 | 232.4s
[💾] Saved best checkpoint → C:\Users\murth\Desktop\nlpSession02\codeBase\translation\artifacts\lstm_attn_best.pt (valid CE=6.1997)
Epoch 03 | train CE: 6.0336 | valid CE: 6.0649 | 230.8s
[💾] Saved best checkpoint → C:\Users\murth\Desktop\nlpSession02\codeBase\translation\artifacts\lstm_attn_best.pt (valid CE=6.0649)
[OK] Training complete. Best valid CE=6.0649


### ✅ Block 9 — LSTM Inference (Greedy) & Evaluation (BLEU + BERTScore)

**Goal.** Load the best LSTM+Attention checkpoint, run **greedy decoding** on the **test** set, and compute **SacreBLEU** and **BERTScore**. Save predictions for later comparison.

**What happens here:**
- Rebuild the same model architecture and **load `artifacts/lstm_attn_best.pt`**.
- Implement greedy decode: start with **BOS**, step until **EOS** or max steps.
- Decode subword IDs back to text with SentencePiece.
- Evaluate on test: **BLEU** (form overlap) and **BERTScore** (semantic).
- Save:
  - `artifacts/lstm_outputs.de-en.jsonl`
  - `artifacts/metrics.lstm.de-en.json`
- Print 5 qualitative examples (SRC / REF / HYP) and the metric summary.


In [10]:
# --- Block 9 Code: LSTM Inference (Greedy) & Evaluation ---
from pathlib import Path
import json, random, math
from typing import List, Dict, Tuple

import torch
import torch.nn as nn
import sentencepiece as spm
import sacrebleu
from bert_score import score as bertscore

# ----------------- Paths & Config -----------------
BASE_DIR = Path(r"C:\Users\murth\Desktop\nlpSession02\codeBase")
PROJ_DIR = BASE_DIR / "translation"
ARTIFACTS = PROJ_DIR / "artifacts"
ckpt_path = ARTIFACTS / "lstm_attn_best.pt"

cfg_path = PROJ_DIR / "mt_config.json"
with open(cfg_path, "r", encoding="utf-8") as f:
    CONFIG = json.load(f)

SEED       = int(CONFIG.get("seed", 42))
LANGPAIR   = CONFIG.get("lang_pair", "de-en")
BPE_VOCAB  = int(CONFIG.get("bpe_vocab", 8000))
MAX_LEN    = int(CONFIG.get("max_len", 40))              # source truncation (already applied in loaders)
GEN_MAX    = int(CONFIG.get("gen_max_len", 60))          # max generated tokens (you can tweak)
HIDDEN     = int(CONFIG.get("lstm_hidden", 256))
LAYERS     = int(CONFIG.get("lstm_layers", 2))
DROPOUT    = float(CONFIG.get("dropout", 0.2))

random.seed(SEED)

# Special token IDs (from Block 6)
ID_UNK = 0
ID_BOS = 1
ID_EOS = 2
ID_PAD = 3

# ----------------- Load tokenizer -----------------
spm_model_path = ARTIFACTS / f"spm_bpe_{BPE_VOCAB}.model"
assert spm_model_path.exists(), f"Missing SentencePiece model: {spm_model_path}"
sp = spm.SentencePieceProcessor(model_file=str(spm_model_path))
VOCAB_SIZE = sp.get_piece_size()

# ----------------- Utilities: data I/O -----------------
def read_jsonl(path: Path):
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

test_path = ARTIFACTS / f"test.{LANGPAIR}.jsonl"
assert test_path.exists(), f"Missing test split: {test_path}"
test_rows = read_jsonl(test_path)

# ----------------- Rebuild Model (same as Block 8) -----------------
device = torch.device("cpu")

class Encoder(nn.Module):
    def __init__(self, vocab_size: int, emb_dim: int, hidden: int, layers: int, dropout: float):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=ID_PAD)
        self.rnn = nn.LSTM(emb_dim, hidden, num_layers=layers, dropout=dropout if layers > 1 else 0.0,
                           bidirectional=True, batch_first=True)
        self.fc_h = nn.Linear(hidden*2, hidden)
        self.fc_c = nn.Linear(hidden*2, hidden)

    def forward(self, src_ids, src_mask):
        x = self.emb(src_ids)             # [B,Ss,E]
        outputs, (h_n, c_n) = self.rnn(x) # outputs: [B,Ss,2H]
        mask = src_mask.unsqueeze(-1)
        summed = (outputs * mask).sum(1)  # [B,2H]
        lens = mask.sum(1).clamp(min=1)   # [B,1]
        mean_ctx = summed / lens
        h0 = torch.tanh(self.fc_h(mean_ctx)).unsqueeze(0).repeat(self.rnn.num_layers//1, 1, 1)
        c0 = torch.tanh(self.fc_c(mean_ctx)).unsqueeze(0).repeat(self.rnn.num_layers//1, 1, 1)
        return outputs, (h0, c0)

class AdditiveAttention(nn.Module):
    def __init__(self, enc_dim: int, dec_dim: int, attn_dim: int):
        super().__init__()
        self.W_h = nn.Linear(enc_dim, attn_dim, bias=False)
        self.W_s = nn.Linear(dec_dim, attn_dim, bias=False)
        self.v   = nn.Linear(attn_dim, 1, bias=False)
    def forward(self, dec_state, enc_outputs, mask):
        Wh = self.W_h(enc_outputs)                         # [B,Ss,A]
        Ws = self.W_s(dec_state).unsqueeze(1)              # [B,1,A]
        scores = self.v(torch.tanh(Wh + Ws)).squeeze(-1)   # [B,Ss]
        scores = scores.masked_fill(mask == 0, -1e9)
        attn = torch.softmax(scores, dim=-1)               # [B,Ss]
        ctx = torch.bmm(attn.unsqueeze(1), enc_outputs).squeeze(1)  # [B,Henc]
        return ctx, attn

class Decoder(nn.Module):
    def __init__(self, vocab_size: int, emb_dim: int, hidden: int, layers: int, dropout: float, enc_out_dim: int):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=ID_PAD)
        self.rnn = nn.LSTM(emb_dim + enc_out_dim, hidden, num_layers=layers, dropout=dropout if layers > 1 else 0.0,
                           batch_first=True)
        self.attn = AdditiveAttention(enc_out_dim, hidden, attn_dim=hidden)
        self.fc_out = nn.Linear(hidden + enc_out_dim, vocab_size)
        self.dropout = nn.Dropout(dropout)
    def step(self, y_prev, h, c, enc_outputs, src_mask):
        # y_prev: [B] token ids
        emb = self.dropout(self.emb(y_prev.unsqueeze(1))).squeeze(1)  # [B,E]
        dec_state = h[-1]                                              # [B,H]
        ctx, _ = self.attn(dec_state, enc_outputs, src_mask)           # [B,Henc]
        rnn_in = torch.cat([emb, ctx], dim=-1).unsqueeze(1)            # [B,1,E+Henc]
        out_t, (h, c) = self.rnn(rnn_in, (h, c))                       # out_t: [B,1,H]
        out_t = out_t.squeeze(1)                                       # [B,H]
        logits = self.fc_out(torch.cat([out_t, ctx], dim=-1))          # [B,V]
        return logits, h, c
    def forward(self, *args, **kwargs):
        raise NotImplementedError("Use step() for incremental decoding.")

class Seq2Seq(nn.Module):
    def __init__(self, vocab_size, emb_dim, hidden, layers, dropout):
        super().__init__()
        self.encoder = Encoder(vocab_size, emb_dim, hidden, layers, dropout)
        self.decoder = Decoder(vocab_size, emb_dim, hidden, layers, dropout, enc_out_dim=hidden*2)

# Instantiate & load
EMB_DIM = HIDDEN
model = Seq2Seq(VOCAB_SIZE, EMB_DIM, HIDDEN, LAYERS, DROPOUT).to(device)
ckpt = torch.load(ckpt_path, map_location=device)
model.load_state_dict(ckpt["model_state"])
model.eval()
print(f"[OK] Loaded checkpoint: {ckpt_path}")

# ----------------- Greedy decoding -----------------
@torch.no_grad()
def greedy_decode_one(src_text: str) -> str:
    # Encode source (truncate)
    src_ids = sp.encode(src_text, out_type=int)[:MAX_LEN]
    if not src_ids:  # avoid empty
        src_ids = [ID_UNK]
    src = torch.tensor(src_ids, dtype=torch.long, device=device).unsqueeze(0)  # [1,Ss]
    src_mask = (src != ID_PAD).long()                                          # [1,Ss]

    # Encode
    enc_outputs, (h, c) = model.encoder(src, src_mask)                         # enc_outputs: [1,Ss,2H]

    # Decode step-by-step
    y = torch.tensor([ID_BOS], dtype=torch.long, device=device)                # [1]
    hyp_ids: List[int] = []
    for _ in range(GEN_MAX):
        logits, h, c = model.decoder.step(y, h, c, enc_outputs, src_mask)      # logits: [1,V]
        next_id = int(torch.argmax(logits, dim=-1).item())
        if next_id == ID_EOS:
            break
        hyp_ids.append(next_id)
        y = torch.tensor([next_id], dtype=torch.long, device=device)

    # Decode IDs → text
    return sp.decode(hyp_ids)

# ----------------- Batch evaluation on TEST -----------------
pred_rows = []
for r in test_rows:
    hyp = greedy_decode_one(r["src"])
    pred_rows.append({"src": r["src"], "ref": r["tgt"], "hyp": hyp})

# Save predictions
out_path = ARTIFACTS / f"lstm_outputs.{LANGPAIR}.jsonl"
with out_path.open("w", encoding="utf-8") as f:
    for r in pred_rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print(f"[SAVE] {out_path}  ({len(pred_rows)} lines)")

# ----------------- Metrics -----------------
refs = [r["ref"] for r in pred_rows]
hyps = [r["hyp"] for r in pred_rows]

bleu = sacrebleu.corpus_bleu(hyps, [refs]).score
P, R, F1 = bertscore(hyps, refs, lang="en", rescale_with_baseline=True)
bert_p = float(P.mean()); bert_r = float(R.mean()); bert_f1 = float(F1.mean())

metrics = {
    "model": "lstm_attn_greedy",
    "langpair": LANGPAIR,
    "bleu": bleu,
    "bertscore": {"precision": bert_p, "recall": bert_r, "f1": bert_f1},
    "n_samples": len(pred_rows),
}
metrics_path = ARTIFACTS / f"metrics.lstm.{LANGPAIR}.json"
with metrics_path.open("w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)
print("\n=== LSTM+Attention (Greedy) Metrics ===")
print(f"BLEU:          {bleu:.2f}")
print(f"BERTScore-P:   {bert_p:.4f}")
print(f"BERTScore-R:   {bert_r:.4f}")
print(f"BERTScore-F1:  {bert_f1:.4f}")
print(f"[SAVE] {metrics_path}")

# ----------------- Preview a few translations -----------------
print("\n[PREVIEW] LSTM translations (5 examples)")
for r in random.sample(pred_rows, min(5, len(pred_rows))):
    print("-"*80)
    print("SRC:", r["src"])
    print("REF:", r["ref"])
    print("HYP:", r["hyp"])


[OK] Loaded checkpoint: C:\Users\murth\Desktop\nlpSession02\codeBase\translation\artifacts\lstm_attn_best.pt
[SAVE] C:\Users\murth\Desktop\nlpSession02\codeBase\translation\artifacts\lstm_outputs.de-en.jsonl  (1000 lines)


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



=== LSTM+Attention (Greedy) Metrics ===
BLEU:          0.09
BERTScore-P:   -0.5368
BERTScore-R:   0.0127
BERTScore-F1:  -0.2794
[SAVE] C:\Users\murth\Desktop\nlpSession02\codeBase\translation\artifacts\metrics.lstm.de-en.json

[PREVIEW] LSTM translations (5 examples)
--------------------------------------------------------------------------------
SRC: Ich komme nun zum Ende.
REF: So, I'm going to wrap up now.
HYP: And's's's a.
--------------------------------------------------------------------------------
SRC: Es ist nicht unser Problem.
REF: It's not our problem.
HYP: And's's's's.
--------------------------------------------------------------------------------
SRC: Also war ich ein bisschen skeptisch, was soll ich damit anfangen?
REF: So I was a little bit skeptical, what should I do with it?
HYP: And I's, I, I, I, I, I, I, I, I, I, I, I, I, I, I, I, I, I, I, I, I, I, I, I, I, I's, and's
--------------------------------------------------------------------------------
SRC: Mein Vorsc

### ⚡ Block 10 — Transformer Inference (Pretrained MarianMT) & Evaluation

**Goal.** Use a **pretrained Transformer** (Hugging Face `Helsinki-NLP/opus-mt-de-en`) to translate the same German→English test set.  

**What happens here:**
- Load pretrained MarianMT tokenizer + model (`Helsinki-NLP/opus-mt-de-en`).  
- Run batch translation on test set (≈1k sentences).  
- Save outputs → `artifacts/transformer_outputs.de-en.jsonl`.  
- Compute BLEU + BERTScore (like baseline and LSTM).  
- Print 5 sample translations to showcase fluency.  


In [11]:
# --- Block 10 Code: Transformer Inference & Evaluation ---
from pathlib import Path
import json, random

import sacrebleu
from bert_score import score as bertscore
import torch
from transformers import MarianMTModel, MarianTokenizer

BASE_DIR = Path(r"C:\Users\murth\Desktop\nlpSession02\codeBase")
PROJ_DIR = BASE_DIR / "translation"
ARTIFACTS = PROJ_DIR / "artifacts"
ARTIFACTS.mkdir(parents=True, exist_ok=True)

cfg_path = PROJ_DIR / "mt_config.json"
with open(cfg_path, "r", encoding="utf-8") as f:
    CONFIG = json.load(f)
LANGPAIR = CONFIG.get("lang_pair", "de-en")

test_path = ARTIFACTS / f"test.{LANGPAIR}.jsonl"
assert test_path.exists(), f"Missing test set: {test_path}"

# ----------------- Load pretrained MarianMT -----------------
model_name = "Helsinki-NLP/opus-mt-de-en"
print(f"[LOAD] MarianMT {model_name}")
tokenizer = MarianTokenizer.from_pretrained(model_name)
model = MarianMTModel.from_pretrained(model_name)
device = torch.device("cpu")  # stick to CPU for classroom
model = model.to(device)
model.eval()

# ----------------- Load test rows -----------------
rows = []
with test_path.open("r", encoding="utf-8") as f:
    for line in f:
        rows.append(json.loads(line))
print(f"[TEST] {len(rows)} sentences")

# ----------------- Batch translate -----------------
pred_rows = []
batch_size = 16
for i in range(0, len(rows), batch_size):
    batch = rows[i:i+batch_size]
    src_texts = [r["src"] for r in batch]
    inputs = tokenizer(src_texts, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)
    with torch.no_grad():
        translated = model.generate(**inputs, max_length=128)
    outs = tokenizer.batch_decode(translated, skip_special_tokens=True)
    for r, hyp in zip(batch, outs):
        pred_rows.append({"src": r["src"], "ref": r["tgt"], "hyp": hyp})

# Save predictions
out_path = ARTIFACTS / f"transformer_outputs.{LANGPAIR}.jsonl"
with out_path.open("w", encoding="utf-8") as f:
    for r in pred_rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
print(f"[SAVE] {out_path}")

# ----------------- Metrics -----------------
refs = [r["ref"] for r in pred_rows]
hyps = [r["hyp"] for r in pred_rows]

bleu = sacrebleu.corpus_bleu(hyps, [refs]).score
P, R, F1 = bertscore(hyps, refs, lang="en", rescale_with_baseline=True)
bert_p = float(P.mean()); bert_r = float(R.mean()); bert_f1 = float(F1.mean())

metrics = {
    "model": "transformer_marianmt",
    "langpair": LANGPAIR,
    "bleu": bleu,
    "bertscore": {"precision": bert_p, "recall": bert_r, "f1": bert_f1},
    "n_samples": len(pred_rows),
}
metrics_path = ARTIFACTS / f"metrics.transformer.{LANGPAIR}.json"
with metrics_path.open("w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

print("\n=== Transformer (MarianMT) Metrics ===")
print(f"BLEU:          {bleu:.2f}")
print(f"BERTScore-P:   {bert_p:.4f}")
print(f"BERTScore-R:   {bert_r:.4f}")
print(f"BERTScore-F1:  {bert_f1:.4f}")
print(f"[SAVE] {metrics_path}")

# ----------------- Preview -----------------
print("\n[PREVIEW] Transformer translations (5 examples)")
for r in random.sample(pred_rows, min(5, len(pred_rows))):
    print("-"*80)
    print("SRC:", r["src"])
    print("REF:", r["ref"])
    print("HYP:", r["hyp"])


[LOAD] MarianMT Helsinki-NLP/opus-mt-de-en


tokenizer_config.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

c:\Users\murth\Desktop\nlpSession02\codeBase\.venv\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\murth\.cache\huggingface\hub\models--Helsinki-NLP--opus-mt-de-en. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


source.spm:   0%|          | 0.00/797k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/768k [00:00<?, ?B/s]

...

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/298M [00:00<?, ?B/s]

[SAVE] C:\Users\murth\Desktop\nlpSession02\codeBase\translation\artifacts\transformer_outputs.de-en.jsonl


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



=== Transformer (MarianMT) Metrics ===
BLEU:          38.53
BERTScore-P:   0.7443
BERTScore-R:   0.7128
BERTScore-F1:  0.7286
[SAVE] C:\Users\murth\Desktop\nlpSession02\codeBase\translation\artifacts\metrics.transformer.de-en.json

[PREVIEW] Transformer translations (5 examples)
--------------------------------------------------------------------------------
SRC: Man erkennt die zentralen Figuren, etwa wer die Anführer der Gruppe sind.
REF: You can see the hubs, like who are the leaders in the group.
HYP: You can recognize the central figures, like who the leaders of the group are.
--------------------------------------------------------------------------------
SRC: Das Geheimnis ist, dass das Spiel der Schlüssel zu diesen Fertigkeiten ist.
REF: The secret is that play is the key to these capacities.
HYP: The secret is that the game is the key to these skills.
--------------------------------------------------------------------------------
SRC: Wir wiederholten diese Übung mit denselb

### 📈 Block 11 — Compare Metrics Across Models (Baseline vs LSTM vs Transformer)

**Goal.** Aggregate the saved metrics and show a single comparison table so students see the progression clearly.

**What happens here:**
- Load:
  - `artifacts/metrics.baseline.de-en.json`
  - `artifacts/metrics.lstm.de-en.json`
  - `artifacts/metrics.transformer.de-en.json`
- Print a compact table (BLEU, BERTScore-F1).
- Save a CSV `artifacts/metrics_comparison.de-en.csv` for your slides.


In [1]:
# --- Block 11 Code: Metrics Comparison Table ---
from pathlib import Path
import json
import csv

BASE_DIR = Path(r"C:\Users\murth\Desktop\nlpSession02\codeBase")
PROJ_DIR = BASE_DIR / "translation"
ARTIFACTS = PROJ_DIR / "artifacts"

cfg_path = PROJ_DIR / "mt_config.json"
with open(cfg_path, "r", encoding="utf-8") as f:
    CONFIG = json.load(f)
LANGPAIR = CONFIG.get("lang_pair", "de-en")

paths = {
    "Pre-Neural (Word Dict)": ARTIFACTS / f"metrics.baseline.{LANGPAIR}.json",
    "LSTM+Attention (Greedy)": ARTIFACTS / f"metrics.lstm.{LANGPAIR}.json",
    "Transformer (MarianMT)":  ARTIFACTS / f"metrics.transformer.{LANGPAIR}.json",
}

rows = []
for name, p in paths.items():
    if not p.exists():
        print(f"[WARN] Missing metrics file: {p} (skip)")
        continue
    with p.open("r", encoding="utf-8") as f:
        m = json.load(f)
    rows.append({
        "Model": name,
        "BLEU": float(m.get("bleu", 0.0)),
        "BERTScore-F1": float(m.get("bertscore", {}).get("f1", 0.0)),
        "Samples": int(m.get("n_samples", 0)),
    })

# Pretty print table
if rows:
    print("\n=== Metrics Comparison ===")
    print(f"{'Model':35} | {'BLEU':>6} | {'BERT-F1':>8} | {'N':>6}")
    print("-"*65)
    for r in rows:
        print(f"{r['Model']:35} | {r['BLEU']:6.2f} | {r['BERTScore-F1']:8.4f} | {r['Samples']:6d}")
else:
    print("[INFO] No metrics found. Run Blocks 5, 9, and 10 first.")

# Save CSV for slides
csv_path = ARTIFACTS / f"metrics_comparison.{LANGPAIR}.csv"
with csv_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["Model","BLEU","BERTScore-F1","Samples"])
    writer.writeheader()
    for r in rows:
        writer.writerow(r)
print(f"[SAVE] {csv_path}")



=== Metrics Comparison ===
Model                               |   BLEU |  BERT-F1 |      N
-----------------------------------------------------------------
Pre-Neural (Word Dict)              |   1.90 |  -0.0344 |   1000
LSTM+Attention (Greedy)             |   0.09 |  -0.2794 |   1000
Transformer (MarianMT)              |  38.53 |   0.7286 |   1000
[SAVE] C:\Users\murth\Desktop\nlpSession02\codeBase\translation\artifacts\metrics_comparison.de-en.csv


### 🚦 Key Observation

- **Baseline (word-for-word SMT-like)**  
  - BLEU ≈ 2  
  - Some words match, but poor ordering → slightly better than chance.  

- **LSTM+Attention (trained from scratch, tiny slice, CPU)**  
  - BLEU ≈ 0.5  
  - Model underfits badly; shows that neural MT *needs lots of data, GPU, and careful tuning*.  
  - Excellent example of *why Transformers displaced RNNs quickly*.  

- **Transformer (pretrained T5/MarianMT)**  
  - BLEU ≈ 38  
  - Fluency and semantic accuracy vastly superior.  
  - Demonstrates power of large-scale pretraining + self-attention.  
